# Goal

The goal of this project is to perform exploratory data analysis on the IEEE-CIS Fraud Detection dataset and identify patterns that may help separate fraudulent from non-fraudulent transactions.

This is a medium-to-large, highly imbalanced tabular dataset with hundreds of anonymized features. The project focuses on:

- memory-aware data handling;
- missingness analysis;
- fraud-rate analysis by interpretable feature groups;
- identifying useful preprocessing decisions for later modeling.

This notebook is focused on Vesta Features Exploration. Modeling, experiment tracking, SHAP analysis, and deployment will be handled in separate notebooks/modules.

Run this notebook in Google Colab with Kaggle credentials in Colab secrets. The exported feature list is learned from the earlier 80% of transactions only.

In [1]:
# Pin the XGBoost version used for the Colab feature-selection artifacts.
import importlib.metadata
import subprocess
import sys
try:
    installed_xgb = importlib.metadata.version("xgboost")
except importlib.metadata.PackageNotFoundError:
    installed_xgb = None
if installed_xgb != "3.4.1":
    if "xgboost" in sys.modules:
        raise RuntimeError("Restart the Colab runtime before installing XGBoost 3.4.1")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost==3.4.1"])
import xgboost as xgb
assert xgb.__version__ == "3.4.1"
print("Feature-selection XGBoost:", xgb.__version__)

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import plotly.express as px

try:
    from google.colab import userdata
    key = userdata.get("KAGGLE_API_TOKEN")
    if key:
        os.environ["KAGGLE_API_TOKEN"] = key.strip()
except ImportError:
    pass

Feature-selection XGBoost: 3.4.1


In [2]:
# Install the Kaggle CLI only when this Colab runtime needs it.
import importlib.metadata
import subprocess
import sys

try:
    kaggle_version = importlib.metadata.version("kaggle")
except importlib.metadata.PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    kaggle_version = importlib.metadata.version("kaggle")
print("Kaggle CLI:", kaggle_version)

Kaggle CLI: 2.0.2


# Helper functions

In [3]:
import warnings
from contextlib import nullcontext

def reduce_memory(df: pd.DataFrame, verbose: bool = True, ignore_warnings: bool = False, inplace: bool = True) -> pd.DataFrame:
    """
    Downcast numeric columns to reduce DataFrame memory usage.
    """
    if not inplace:
        df = df.copy()

    start_mem = df.memory_usage(deep=True).sum() / 1e6

    warning_context = warnings.catch_warnings() if ignore_warnings else nullcontext()

    with warning_context:
        if ignore_warnings:
            warnings.filterwarnings(
                "ignore",
                category=RuntimeWarning,
                message="overflow encountered in cast",
            )

        for col in df.columns:
            series = df[col]

            if pd.api.types.is_integer_dtype(series):
                c_min = series.min()
                c_max = series.max()

                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df[col] = series.astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df[col] = series.astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df[col] = series.astype(np.int32)
                else:
                    df[col] = series.astype(np.int64)

            elif pd.api.types.is_float_dtype(series):
                c_min = series.min()
                c_max = series.max()

                if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                    df[col] = series.astype(np.float32)
                else:
                    df[col] = series.astype(np.float64)

    end_mem = df.memory_usage(deep=True).sum() / 1e6

    if verbose:
        reduction = 100 * (start_mem - end_mem) / start_mem if start_mem else 0
        print(f"Memory: {start_mem:.2f} MB → {end_mem:.2f} MB ({reduction:.1f}% reduction)")

    return df

# Data load

In [4]:
import subprocess

archive = Path("/content/ieee-fraud-detection.zip")
required_csvs = [Path("/content/train_transaction.csv"), Path("/content/train_identity.csv")]
if not all(path.exists() for path in required_csvs):
    if not archive.exists():
        subprocess.run(
            ["kaggle", "competitions", "download", "-c", "ieee-fraud-detection", "-p", "/content"],
            check=True,
        )
    subprocess.run(["unzip", "-qo", str(archive), "-d", "/content"], check=True)
if not all(path.exists() for path in required_csvs):
    raise FileNotFoundError("Kaggle training CSVs are missing from /content")
print("Training CSVs ready in /content")


Training CSVs ready in /content


In [5]:
BASE_PATH = Path("/content")
TRAIN_TRANSACTION = BASE_PATH / "train_transaction.csv"
TRAIN_IDENTITY = BASE_PATH / "train_identity.csv"

# V ranking uses transaction time, label, and V columns only.
transaction_columns = pd.read_csv(TRAIN_TRANSACTION, nrows=0).columns
v_features_all = [col for col in transaction_columns if col.startswith("V")]
train_df_merged = reduce_memory(pd.read_csv(
    TRAIN_TRANSACTION,
    usecols=["TransactionDT", "isFraud"] + v_features_all,
), True)
print("Loaded", len(train_df_merged), "transactions and", len(v_features_all), "V features")


Memory: 1610.99 MB → 803.73 MB (50.1% reduction)
Loaded 590540 transactions and 339 V features


Only the transaction time, label, and V columns are loaded. Numeric
columns are downcast once as the CSV is read. The latest 20% of transactions
remains available only for checking the time boundary; no holdout values or
labels enter the ranker.


# Vesta-engineered V-features

In [6]:
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Reserve the latest 20% of transactions for final modeling.
time_order = train_df_merged["TransactionDT"].sort_values(kind="stable").index
train_period_idx = time_order[:int(len(time_order) * 0.8)]
final_holdout_idx = time_order[int(len(time_order) * 0.8):]
X = train_df_merged.loc[train_period_idx, v_features_all].astype("float32")
y = train_df_merged.loc[train_period_idx, "isFraud"]
inner_cutoff = int(len(train_period_idx) * 0.8)
selection_train_idx = train_period_idx[:inner_cutoff]
selection_val_idx = train_period_idx[inner_cutoff:]
X_train_v, X_val_v = X.loc[selection_train_idx], X.loc[selection_val_idx]
y_train_v, y_val_v = y.loc[selection_train_idx], y.loc[selection_val_idx]
assert train_df_merged.loc[train_period_idx, "TransactionDT"].max() <= train_df_merged.loc[final_holdout_idx, "TransactionDT"].min()
print("Exploration rows:", len(selection_train_idx), "inner validation rows:", len(selection_val_idx))
print("Final holdout rows excluded from feature selection:", len(final_holdout_idx))

scale_pos_weight_value = (len(y_train_v) - y_train_v.sum()) / y_train_v.sum()
model = xgb.XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", n_estimators=500,
    learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=42, scale_pos_weight=scale_pos_weight_value,
    importance_type="gain", tree_method="hist"
)
model.fit(X_train_v, y_train_v)
y_val_proba = model.predict_proba(X_val_v)[:, 1]
print("Inner validation ROC-AUC:", roc_auc_score(y_val_v, y_val_proba))
print("Inner validation PR-AUC:", average_precision_score(y_val_v, y_val_proba))
importance_df = pd.DataFrame({"feature": X_train_v.columns, "importance_gain": model.feature_importances_}).sort_values("importance_gain", ascending=False, kind="stable")
importance_df.head(20)


Exploration rows: 377945 inner validation rows: 94487
Final holdout rows excluded from feature selection: 118108
Inner validation ROC-AUC: 0.8509664919281921
Inner validation PR-AUC: 0.4606121549078217


,feature,importance_gain
257,V258,0.088540
69,V70,0.054649
200,V201,0.051617
217,V218,0.045121
293,V294,0.036016
93,V94,0.031895
90,V91,0.030980
186,V187,0.027822
29,V30,0.016715
231,V232,0.016185


XGBoost gain ranks V features on the first 64% of transactions. The next 16% is used only for exploratory diagnostics. The final 20% time holdout is untouched until notebooks 4 and 5. Permutation importance and correlations below are diagnostics, not inputs to the exported feature list.

Because permutation importance is computationally expensive on the full set of Vesta-engineered features, I first use XGBoost gain importance to select the strongest candidate features, then run permutation importance only on the top-ranked subset.

In [7]:

top_v_features = importance_df.head(30)["feature"].tolist()

model_top_v = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    scale_pos_weight=scale_pos_weight_value,
    importance_type="gain"
)

model_top_v.fit(X_train_v[top_v_features], y_train_v)

from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    model_top_v,
    X_val_v[top_v_features],
    y_val_v,
    scoring="average_precision",
    n_repeats=3,
    random_state=42,
    n_jobs=-1
)

perm_importance_df = pd.DataFrame({
    "feature": top_v_features,
    "permutation_importance": perm_result.importances_mean,
    "std": perm_result.importances_std
}).sort_values("permutation_importance", ascending=False)

perm_importance_df.head(20)

,feature,permutation_importance,std
12,V45,0.022625,0.000065
10,V308,0.018019,0.000451
29,V67,0.017963,0.000342
23,V261,0.017367,0.000940
0,V258,0.016750,0.000566
15,V283,0.016373,0.000241
5,V94,0.015391,0.000903
3,V218,0.014515,0.001372
1,V70,0.010603,0.001076
2,V201,0.009280,0.000522


In [8]:
# Correlation diagnostics use only the exploration training period.
import pandas as pd
v_features_and_target = [col for col in train_df_merged.columns if col.startswith("V")] + ["isFraud"]
df_for_corr = train_df_merged.loc[selection_train_idx, v_features_and_target].copy()
correlation_matrix = df_for_corr.corr(numeric_only=True)
is_fraud_correlations = correlation_matrix["isFraud"].drop("isFraud")
top_15_positive = is_fraud_correlations.sort_values(ascending=False).head(15)
top_15_negative = is_fraud_correlations.sort_values().head(15)
print("Top positive correlations (inner training only):")
print(top_15_positive)
print("Top negative correlations (inner training only):")
print(top_15_negative)


Top positive correlations (inner training only):
V257    0.357149
V244    0.346089
V242    0.339654
V246    0.335132
V201    0.328027
V200    0.311996
V189    0.308192
V188    0.304027
V233    0.296818
V158    0.282258
V156    0.281606
V149    0.279462
V231    0.272999
V45     0.272043
V217    0.269374
Name: isFraud, dtype: float64
Top negative correlations (inner training only):
V48    -0.111957
V90    -0.109410
V29    -0.108303
V49    -0.107676
V91    -0.105573
V69    -0.105350
V30    -0.104975
V70    -0.102401
V10    -0.079570
V11    -0.077902
V150   -0.070385
V159   -0.068655
V160   -0.068006
V151   -0.067808
V145   -0.066320
Name: isFraud, dtype: float64


Gain, permutation importance, and correlation are exploratory diagnostics here. Their rankings can differ because V features are transformed and correlated. The exported feature list below is generated only from gain importance fitted on the full earlier 80% training period. No labels or feature values from the final 20% holdout enter that fit.

In [9]:
# A compact exploratory subset from inner-training gain, not a frozen list from the old run.
exploratory_v_features = importance_df.head(19)["feature"].tolist()


In [10]:
model_selected_v = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    scale_pos_weight=scale_pos_weight_value,
    importance_type="gain"
)

model_selected_v.fit(X_train_v[exploratory_v_features], y_train_v)

y_val_proba_selected = model_selected_v.predict_proba(
    X_val_v[exploratory_v_features]
)[:, 1]

print("Selected V ROC-AUC:", roc_auc_score(y_val_v, y_val_proba_selected))
print("Selected V PR-AUC:", average_precision_score(y_val_v, y_val_proba_selected))

Selected V ROC-AUC: 0.8424708426418976
Selected V PR-AUC: 0.4442070939319183


How much signal does a small V-feature subset retain on the inner validation period? This is a diagnostic only; it does not choose the exported feature count.


In [11]:
results = []

for k in [20, 30, 50, 75, 100]:
    top_features = importance_df.head(k)["feature"].tolist()

    model_k = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=42,
        scale_pos_weight=scale_pos_weight_value,
        importance_type="gain"
    )

    model_k.fit(X_train_v[top_features], y_train_v)

    y_val_proba_k = model_k.predict_proba(X_val_v[top_features])[:, 1]

    results.append({
        "num_features": k,
        "roc_auc": roc_auc_score(y_val_v, y_val_proba_k),
        "pr_auc": average_precision_score(y_val_v, y_val_proba_k)
    })

feature_count_results = pd.DataFrame(results)
feature_count_results

,num_features,roc_auc,pr_auc
0,20,0.842738,0.443334
1,30,0.844955,0.448135
2,50,0.849319,0.447536
3,75,0.849707,0.449650
4,100,0.849141,0.450325


The top-k sweep above is an inner-validation diagnostic. Its scores do not determine the exported feature count. The final count is selected by a predeclared 95% cumulative gain rule fitted only on the earlier 80% training period.

The inner-validation sweep illustrates the tradeoff between a smaller V-feature set and predictive signal. Rerun it to obtain current scores; the exported list is selected separately from training-period gain.


# Conclusion

The final V-feature list is generated from a supervised XGBoost ranking fitted on the earlier 80% of transactions. We retain the smallest prefix of the gain ranking that accounts for at least 95% of total positive gain. This rule uses no final-holdout rows or labels. The earlier notebook's top-75 list was affected by a random split of the full data and should not be reused.

In [12]:
importance_df.to_csv("v_feature_importance_gain_inner_train.csv", index=False)
perm_importance_df.to_csv("v_feature_permutation_importance_inner_validation.csv", index=False)
feature_count_results.to_csv("v_feature_count_results_inner_validation.csv", index=False)

In [13]:
import json
import numpy as np

# Refit on the entire earlier 80% only; match the Kedro selector settings.
X_selection_v = X
y_selection_v = y
selection_weight = (len(y_selection_v) - y_selection_v.sum()) / y_selection_v.sum()
final_v_ranker = xgb.XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", n_estimators=500,
    learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=42, scale_pos_weight=selection_weight,
    importance_type="gain", tree_method="hist"
)
final_v_ranker.fit(X_selection_v, y_selection_v)
final_v_importance = pd.Series(final_v_ranker.feature_importances_, index=v_features_all).sort_values(ascending=False, kind="stable")
if final_v_importance.sum() <= 0:
    raise ValueError("No positive V-feature gain")
selected_count = int(np.searchsorted((final_v_importance.cumsum() / final_v_importance.sum()).to_numpy(), 0.95, side="left")) + 1
selected_v_features = final_v_importance.head(selected_count).index.tolist()
import hashlib
feature_sha256 = hashlib.sha256(json.dumps(selected_v_features, separators=(",", ":")).encode()).hexdigest()
# Keep the list produced by this training-period fit. Record its exact digest
# so modeling notebooks use the same list, even if another platform ranks ties differently.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

transaction_csv_sha256 = sha256_file(TRAIN_TRANSACTION)
identity_csv_sha256 = sha256_file(TRAIN_IDENTITY)
selection_metadata = {
    "method": "XGBoost gain importance on chronological training rows only",
    "xgboost_version": xgb.__version__,
    "pandas_version": pd.__version__,
    "feature_sha256": feature_sha256,
    "transaction_csv_sha256": transaction_csv_sha256,
    "identity_csv_sha256": identity_csv_sha256,
    "gain_fraction": 0.95,
    "n_selection_rows": len(train_period_idx),
    "selection_time_max": float(train_df_merged.loc[train_period_idx, "TransactionDT"].max()),
    "holdout_time_min": float(train_df_merged.loc[final_holdout_idx, "TransactionDT"].min()),
    "n_selected_features": len(selected_v_features),
}
with open("selected_v_features.json", "w") as f:
    json.dump(selected_v_features, f, indent=2)
with open("v_feature_selection_metadata.json", "w") as f:
    json.dump(selection_metadata, f, indent=2)
print("Saved", len(selected_v_features), "V features from training-period gain only")


Saved 229 V features from training-period gain only


In [14]:
from google.colab import drive
import shutil
from pathlib import Path

drive.mount('/content/drive')
destination_dir = Path("/content/drive/My Drive/ML/Fraud Detection/Data/Artifacts")
destination_dir.mkdir(parents=True, exist_ok=True)
for filename in ["selected_v_features.json", "v_feature_selection_metadata.json"]:
    shutil.copy(filename, destination_dir / filename)
    print("Saved", destination_dir / filename)


Mounted at /content/drive
Saved /content/drive/My Drive/ML/Fraud Detection/Data/Artifacts/selected_v_features.json
Saved /content/drive/My Drive/ML/Fraud Detection/Data/Artifacts/v_feature_selection_metadata.json


# V-feature summary

The anonymized V features provide signal on the inner time split. The exported
list is ranked by XGBoost gain on the earlier 80% only, with its size set by
the predeclared 95% cumulative-gain rule.


# Modeling plan

1. Sort transactions by `TransactionDT` and reserve the latest 20% as the final time holdout.
2. Use only the earlier 80% for supervised feature selection and preprocessing. Within that period, use an inner time split for exploratory ranking, model tuning, and threshold tuning.
3. Preserve missingness with numeric indicators or explicit missing categories. Treat card/address identifiers, email domains, and device fields as categorical where appropriate.
4. Train a simple baseline, then compare LightGBM and XGBoost using PR-AUC as the primary metric; report ROC-AUC, precision, recall, and F1 as supporting metrics.
5. Evaluate the chosen configuration on the final time holdout once, then interpret the fitted model. A fresh later labeled period is needed for an untouched estimate because this holdout was examined in earlier project runs.

| Finding | Modeling decision |
|---|---|
| Missingness is target-associated | Add missing indicators or preserve missing values as an explicit category. |
| ProductCD fraud rates differ | Encode ProductCD as a categorical feature. |
| DeviceInfo is high-cardinality | Group rare values before encoding. |
| TransactionAmt is skewed | Test `log1p(TransactionAmt)` for linear models. |
| card/addr numeric columns act categorical | Avoid treating all numeric columns as continuous measurements. |
| Dataset is imbalanced | Use PR-AUC, recall, and threshold tuning instead of relying on accuracy. |
| Anonymized C/D/M features show potential signal | Inspect systematically and evaluate them during modeling instead of dropping them only because they lack semantic meaning. |